# Curriculum 05 · Lab 4 — HyDE: embed a hypothetical answer, not the question

**Goal:** Defeat lexical mismatch. The user asks "Who founded Montevideo?"
but the passage says "The city was established by the Spanish in 1726".
Query and passage share almost no tokens, so the question's embedding lands
far from the answer's. HyDE (Hypothetical Document Embeddings) closes the gap
at the QUERY side: an LLM writes a short hypothetical passage that WOULD
answer the question, and the retriever embeds THAT instead of the raw query.

```
Transformer : HyDERetriever (retrieval/hyde.py)
LLM         : llama-3.3-70b-versatile (GroqLLM) — writes the hypothetical
Store       : FAISSVectorStore (in-memory) + SimilarityRetriever
Embedding   : BGE (BAAI/bge-base-en-v1.5, local, CPU)
Data        : rag-mini-wikipedia — first 100 passages, questions 1606/1610/1626
```

**Why HyDE:** a hypothetical passage is written in *source-document language*,
so its embedding lives in the same region of vector space as the real chunks.
The question only needs to be understood once — by the cheap, capable LLM —
while the embedding model only ever sees document-shaped text. Cost: one LLM
call + one extra embed per question, no index rebuild.

This is the fourth lab of track 05-query-transformation (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Installs the lab's dependencies (a no-op if already present),
loads `GROQ_API_KEY` from the repo-root `.env`, and puts the repo-root
component library on `sys.path` so this notebook reuses `retrieval/*.py`,
`llms/groq.py`, `vectordb/faiss.py` and `embeddings/bge.py` exactly like the
lab script.

**WHY:** Everything embeds **locally** with BGE via sentence-transformers —
no API embeddings anywhere. The LLM is only the query-*transformation* step
(Groq's `llama-3.3-70b-versatile`; a commented Gemini alternative is kept in
the source). The retriever classes live in the repo's shared component
library (`retrieval/`), not inside the lab, so the exact same code path runs
here, in the `.py`, and in later tracks.

**Paths:** the next cell resolves the **repo root** automatically — it works
whether the kernel launches from the repo root (like the lab script) or from
the notebook's own folder (the Jupyter default) — and `cd`s into it so every
path stays repo-relative.

**WHAT TO EXPECT:** no output from the pip cell (packages already
installed), a silent import from the second. The BGE model is loaded lazily
when the experiment cell first calls it; the Groq key is read from `.env`.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings (embeddings/bge.py)
#   faiss-cpu             -> the FAISS index (vectordb/faiss.py)
#   langchain-groq        -> GroqLLM (the HyDE generator, llms/groq.py)
#   python-dotenv         -> loads GROQ_API_KEY from the repo-root .env
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers faiss-cpu langchain-groq python-dotenv pandas



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env

from embeddings.bge import BGEEmbedding  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from llms.groq import GroqLLM  # noqa: E402
from retrieval.hyde import HyDERetriever  # noqa: E402
from retrieval.similarity import SimilarityRetriever  # noqa: E402
from vectordb.faiss import FAISSVectorStore  # noqa: E402


## 1 · Configuration — the experiment's knobs

**WHAT:** The corpus constants (`N_PASSAGES = 100`, `QUESTION_IDS =
[1606, 1610, 1626]`, `TOP_K = 3`) plus the Groq model name — the model that
writes each hypothetical passage (a commented Gemini alternative is kept in
the source).

**WHY:** Same three questions as labs 01–02 so the four transformations can
be compared directly — the only variable is the retriever wrapper.


In [3]:
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1626]  # same questions as labs 01–02, for comparison
TOP_K = 3
LLM_MODEL = "llama-3.3-70b-versatile"  # Groq writes the hypothetical passage, never embeds
# (Gemini alternative: LLM_MODEL = "gemini-2.5-flash" — needs GOOGLE_API_KEY in .env)
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
PREVIEW = 62  # max characters of passage text shown next to each hit


## 2 · Load — corpus + questions from the fresh parquet files

**WHAT:** `load_passages` pulls the first `n` passages (text + ids) from
`passages.parquet`; `load_questions` pulls specific rows by id from
`test.parquet`; `preview` flattens a passage for one-line printing.

**WHY:** Identical helpers to labs 01–02 keep the comparison honest — same
corpus, same questions, different transformation.


In [4]:
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 3 · Experiment — raw vs HyDE retrieval for the same questions

**WHAT:** `run_experiment` embeds the 100-passage subset once, builds the
FAISS store, then per question runs the plain `SimilarityRetriever` AND the
`HyDERetriever` over the same store — recording the raw top-k, the LLM's
hypothetical passage, its wall time, and the HyDE top-k.

**WHY:** Both paths share one index; any difference in the top-1 is caused
by embedding a hypothetical passage instead of the query. The hypothetical
is captured separately (one extra `_hypothetical` call) purely for the demo —
the retrieval itself makes exactly one LLM call per question.


In [5]:
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Embed locally (BGE) and index in-memory ---------------------------
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME)
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0

    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    store = FAISSVectorStore(embedding=embedder)
    t0 = time.perf_counter()
    store.add(chunks, embeddings=passage_vecs)
    index_s = time.perf_counter() - t0

    # --- The two retrievers over the SAME store -----------------------------
    raw_retriever = SimilarityRetriever(store, top_k=TOP_K)
    hyde_llm = GroqLLM(model=LLM_MODEL)
    hyde_retriever = HyDERetriever(hyde_llm, raw_retriever, top_k=TOP_K)

    # --- Per question: raw retrieval + the hypothetical passage + retrieval --
    results = []
    for qid, qtext in questions:
        raw_docs = raw_retriever.retrieve(qtext)
        t0 = time.perf_counter()
        hypothetical = hyde_retriever._hypothetical(qtext)
        hyde_s = time.perf_counter() - t0
        hyde_docs = hyde_retriever.retrieve(qtext)
        results.append(
            {
                "qid": qid,
                "question": qtext,
                "hypothetical": hypothetical,
                "hyde_s": hyde_s,
                "raw_docs": raw_docs,
                "hyde_docs": hyde_docs,
            }
        )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "indexed": len(passage_texts),
        "embed_s": embed_s,
        "index_s": index_s,
        "results": results,
    }


## 4 · Run — execute the experiment

**WHAT:** Calls `run_experiment()` — one embed, one index build, all
retrievals (plus the Groq transformation calls) — and keeps the artifact
dict as `exp`.

**WHY:** Everything after this cell (the demo and the verification gate)
reads from this single `exp`, so the printed numbers and the verified
numbers are guaranteed to come from the same run. The LLM calls happen here,
once — the gate cell never re-burns them.


In [6]:
exp = run_experiment()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 5 · Demo — read the artifact

**WHAT:** `print_demo` prints the corpus summary, then per question the
hypothetical passage the LLM wrote, and the top-1 passage of both the raw
and the HyDE path.

**WHY:** Read the hypothetical like a retrieval engineer: it is what the
*embedding model* actually saw — prose that looks like the source corpus.
Where the HyDE top-1 differs from the raw one, you are watching the query
side reach a region the raw question could not.


In [7]:
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 04 — HyDE: embed a hypothetical answer, not the question")
    print(f"{BGE_MODEL_NAME} (local) -> FAISS top-{TOP_K} -> {LLM_MODEL} HyDE")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    embedded in {exp['embed_s']:.2f}s (dim 768), indexed in {exp['index_s']:.3f}s")

    print(f"\n[2] Raw vs HyDE (per question):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["question"]}"')
        print(f"      hypothetical ({r['hyde_s']:.1f}s): {r['hypothetical']!r}")
        print(f"      raw   top-1: {preview(r['raw_docs'][0].page_content)}")
        print(f"      hyde  top-1: {preview(r['hyde_docs'][0].page_content)}")

    print("\n[3] Takeaway")
    print("    HyDE converts the question into document-shaped text before")
    print("    embedding, so the embedding model sees what it is best at.")
    print("    The LLM does the understanding (one cheap call per question);")
    print("    the retriever then searches with a passage that shares the")
    print("    source corpus' vocabulary. Cost: one LLM call + one extra")
    print("    embed per question — no index rebuild, no store changes.")


In [8]:
print_demo(exp)


Lab 04 — HyDE: embed a hypothetical answer, not the question
BAAI/bge-base-en-v1.5 (local) -> FAISS top-3 -> llama-3.3-70b-versatile HyDE

[1] Corpus (deterministic subset, no randomness):
    100 passages (first 100 of 3200, ids 0..99)
    embedded in 14.73s (dim 768), indexed in 0.043s

[2] Raw vs HyDE (per question):

    Q[1606] "Is Uruguay's capital Montevideo?"
      hypothetical (0.5s): "Montevideo has been the capital and largest city of Uruguay since 1828, situated on the north shore of the Río de la Plata, and is home to approximately one-third of the country's population."
      raw   top-1: Montevideo, Uruguay's capital.
      hyde  top-1: Montevideo, Uruguay's capital.

    Q[1610] "Who founded Montevideo?"
      hypothetical (0.3s): 'The city of Montevideo was founded in 1726 by Bruno Mauricio de Zabala, a Spanish governor who established the settlement as a fortified port to counter Portuguese expansion in the region.'
      raw   top-1: Uruguay's capital, Montevideo, wa

## 6 · Verification gate — the same checks the .py runs

**WHAT:** Runs the exact `verify_gate`: exactly `N_PASSAGES` indexed, every
question returning `TOP_K` hits on both paths, every hypothetical non-empty
and genuinely *prose* (>= 8 words — a question turned into a passage, not a
paraphrase), and the content checks — the HyDE top-3 must still carry the
answer's keyword (montevideo / spanish / 1930).

**WHY:** `python 04-hyde.py --verify` must print 12/12 PASS; this cell
proves the notebook reproduces the verified `.py` exactly. Keyword checks are
pinned to retrieval outcomes, not exact LLM wording, so the gate is stable.


In [9]:
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Structural properties (no LLM involved).
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))
    checks.append(("each question returns TOP_K raw hits",
                   all(len(r["raw_docs"]) == TOP_K for r in exp["results"])))
    checks.append(("each question returns TOP_K HyDE hits",
                   all(len(r["hyde_docs"]) == TOP_K for r in exp["results"])))

    # The hypothetical passage must be a real passage — non-empty and longer
    # than a bare question (the LLM expands a question into document prose).
    for r in exp["results"]:
        tag = f"Q{r['qid']}"
        checks.append((f"{tag} hypothetical is non-empty",
                       bool(r["hypothetical"].strip())))
        checks.append((f"{tag} hypothetical is prose (>= 8 words), not a query",
                       len(r["hypothetical"].split()) >= 8))

    # Content checks: the HyDE path must still surface the answer's keyword.
    # Q1606 -> Montevideo; Q1610 -> the Spanish; Q1626 -> 1930.
    for r in exp["results"]:
        tag = f"Q{r['qid']}"
        joined = " ".join(d.page_content for d in r["hyde_docs"]).lower()
        if r["qid"] == 1606:
            kw = "montevideo"
        elif r["qid"] == 1610:
            kw = "spanish"
        else:  # 1626
            kw = "1930"
        checks.append((f"{tag} HyDE top-{TOP_K} retains '{kw}'", kw in joined))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


In [10]:
verify_gate(exp)


verification gate:
  [PASS] exactly 100 passages indexed
  [PASS] each question returns TOP_K raw hits
  [PASS] each question returns TOP_K HyDE hits
  [PASS] Q1606 hypothetical is non-empty
  [PASS] Q1606 hypothetical is prose (>= 8 words), not a query
  [PASS] Q1610 hypothetical is non-empty
  [PASS] Q1610 hypothetical is prose (>= 8 words), not a query
  [PASS] Q1626 hypothetical is non-empty
  [PASS] Q1626 hypothetical is prose (>= 8 words), not a query
  [PASS] Q1606 HyDE top-3 retains 'montevideo'
  [PASS] Q1610 HyDE top-3 retains 'spanish'
  [PASS] Q1626 HyDE top-3 retains '1930'


0